# Gender leakage probe on model representations

In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

import helpers

DATA_PATH = '../data'
DB_FILE   = 'FairCVdb.npy'

# Configuration
HIDDEN_LAYERS = (40, 20)
MAX_ITER      = 500
SEED          = 32

SEEDS = range(32, 37)
CONDITIONS = ['none', 'tabular', 'tabular-text', 'tabular-image', 'tabular-image-text']

In [ ]:
fairCV = np.load(os.path.join(DATA_PATH, DB_FILE), allow_pickle=True).item()

profiles_train = fairCV['Profiles Train']
profiles_test  = fairCV['Profiles Test']

y_train = profiles_train[:, 1]
y_test  = profiles_test[:, 1]

male_count = np.count_nonzero(y_test == 0)
female_count = np.count_nonzero(y_test == 1)
total = y_test.size

print("male count:", male_count)
print("female count:", female_count)
print("total count:", total)

chance = max(male_count/total, female_count/total)

In [ ]:
def probe(X_train, X_test, y_train, y_test):
    """
    Predict gender from one input representation usign a classifier.

    Returns test-set accuracy. Close to 0.5 means the representation carries little
    recoverable gender information; close to 1.0 means gender is fully recoverable.

    We're only measuring recoverability, a number close to 1.0 does not mean the model will definitely use the gender information. It just means the information is available to be potentially used.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    clf = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        max_iter=MAX_ITER,
        random_state=SEED,
    )
    clf.fit(X_train_scaled, y_train)

    return accuracy_score(y_test, clf.predict(X_test_scaled))

In [ ]:
class EarlyFusion(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 40)
        self.fc2 = nn.Linear(40, 20)
        self.fc3 = nn.Linear(20, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))


class MidFusion(nn.Module):
    def __init__(self, tabular_size, text_size, image_size, text_width):
        super().__init__()
        self.tabular = nn.Linear(tabular_size, tabular_size)
        self.text = nn.Linear(text_size, text_width)
        self.image = nn.Linear(image_size, image_size)

        self.fc1 = nn.Linear(tabular_size + text_width + image_size, 40)
        self.fc2 = nn.Linear(40, 20)
        self.fc3 = nn.Linear(20, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, tabular, text, image):
        # each modality gets its own layer
        tabular = torch.relu(self.tabular(tabular))
        text = torch.relu(self.text(text))
        image = torch.relu(self.image(image))
        cat = torch.cat((tabular, text, image), 1)

        cat = torch.relu(self.fc1(cat))
        cat = self.dropout(cat)
        cat = torch.relu(self.fc2(cat))
        return torch.sigmoid(self.fc3(cat))


class LateFusion(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 40)
        self.fc2 = nn.Linear(40, 20)
        self.fc3 = nn.Linear(20, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

source: https://web.stanford.edu/~nanbhas/blog/forward-hooks-pytorch/, https://apxml.com/courses/pytorch-for-tensorflow-developers/chapter-6-advanced-pytorch-features-tf-users/pytorch-hooks-functionality

In [ ]:
def grab_last(model, data):
    activations = {}

    model.to(helpers.DEVICE)

    tensors = [torch.FloatTensor(x).to(helpers.DEVICE) for x in data]

    def pre_hook(module, args):
        activations['fc3'] = args[0].detach()

    handle = model.fc3.register_forward_pre_hook(pre_hook)

    model(*tensors)

    handle.remove()
    return activations['fc3'].cpu().numpy() # because probe() expects numpy arrays

In [ ]:
## EARLY FUSION

rows = []

for condition in CONDITIONS:
    (labels_train_biased, labels_train_blind,
     labels_test_biased, labels_test_blind,
     tabular_train, tabular_test,
     text_train, text_test,
     faces_train, faces_test) = helpers.build_features(condition)

    train_concatenated = np.concatenate([tabular_train, text_train, faces_train], axis=1)
    test_concatenated = np.concatenate([tabular_test, text_test, faces_test], axis=1)



    for seed in SEEDS:
        _, model = helpers.train_and_evaluate(
            lambda: EarlyFusion(train_concatenated.shape[1]), # lambda bcs we seed before constructing the model
            [train_concatenated],
            [test_concatenated],
            labels_train_biased,
            seed,
        )

        train_activations = grab_last(model, [train_concatenated])
        test_activations = grab_last(model, [test_concatenated])
        accuracy = probe(train_activations, test_activations, y_train, y_test)

        rows.append({
            'condition': condition,
            'fusion': 'early',
            'branch': 'all',
            'seed': seed,
            'accuracy': accuracy,
        })

    print(condition, 'done')

In [ ]:
## LATE FUSION

for condition in CONDITIONS:
    (labels_train_biased, labels_train_blind,
     labels_test_biased, labels_test_blind,
     tabular_train, tabular_test,
     text_train, text_test,
     faces_train, faces_test) = helpers.build_features(condition)

    modalities = [
        ('tabular', tabular_train, tabular_test),
        ('text', text_train, text_test),
        ('image', faces_train, faces_test),
    ]

    for seed in SEEDS:
        for name, train, test in modalities:
            _, model = helpers.train_and_evaluate(
                lambda: LateFusion(train.shape[1]),
                [train],
                [test],
                labels_train_biased,
                seed,
            )

            train_activations = grab_last(model, [train])
            test_activations = grab_last(model, [test])
            accuracy = probe(train_activations, test_activations, y_train, y_test)

            rows.append({
                'condition': condition,
                'fusion': 'late',
                'branch': name,
                'seed': seed,
                'accuracy': accuracy,
            })

    print(condition, 'done')

In [ ]:
for condition in CONDITIONS:
    (labels_train_biased, labels_train_blind, labels_test_biased, labels_test_blind,
     tabular_train, tabular_test, text_train, text_test, faces_train, faces_test) = helpers.build_features(condition)

    for seed in SEEDS:
        _, model = helpers.train_and_evaluate(
            lambda: MidFusion(tabular_train.shape[1], text_train.shape[1], faces_train.shape[1], 50),
            [tabular_train, text_train, faces_train],
            [tabular_test, text_test, faces_test],
            labels_train_biased,
            seed,
        )

        train_activations = grab_last(model, [tabular_train, text_train, faces_train])
        test_activations = grab_last(model, [tabular_test, text_test, faces_test])
        accuracy = probe(train_activations, test_activations, y_train, y_test)


        rows.append({
            'condition': condition,
            'fusion': 'mid',
            'branch': 'all',
            'seed': seed,
            'accuracy': accuracy,
        })

    print(condition, 'done')

results = pd.DataFrame(rows)
results.to_csv('../results/probe_internal.csv', index=False)
results.groupby(['fusion', 'branch', 'condition'], sort=False)[['accuracy']].agg(['mean', 'std']) # because we run 5 seeds, we aggregate results usign mean + std